# LSTM Final Selected Model


## 1. Load LSTM-Ready Cache


In [ ]:
# Purpose: Loads the optional LSTM-ready cache. This notebook does not scan audio folders,
# make a new split, or call Librosa MFCC extraction.
import json
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from tensorflow.keras.layers import Dense, Dropout, Input, LSTM
from tensorflow.keras.models import Sequential

try:
    from IPython.display import display
except Exception:
    display = print

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

CLASS_NAMES = {0: "bona_fide", 1: "synthetic"}
EPOCHS = 30
BATCH_SIZE = 64
THRESHOLD = 0.5


def resolve_project_root():
    explicit = os.environ.get("INTRO_AI_PROJECT_ROOT")
    if explicit:
        return Path(explicit).expanduser().resolve()
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "Model Variants").exists():
            return candidate
        if (candidate / "Training" / "Model Variants").exists():
            return candidate / "Training"
    return Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701")


PROJECT_ROOT = resolve_project_root()
SHARED_CLASS_WEIGHT_PATH = PROJECT_ROOT / "outputs" / "shared" / "class_weights.json"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "lstm_optional"
TABLES_DIR = OUTPUT_DIR / "tables"
CACHE_DIR = OUTPUT_DIR / "cache"
MODELS_DIR = OUTPUT_DIR / "models"
METRICS_DIR = OUTPUT_DIR / "metrics"
FIGURES_DIR = OUTPUT_DIR / "figures"
for directory in [OUTPUT_DIR, TABLES_DIR, CACHE_DIR, MODELS_DIR, METRICS_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found: {path}. Run the analysis notebook, then 00_LSTM_Data_Preparation.ipynb."
        )
    return path


def load_shared_class_weights(path):
    with open(require_file(path), "r", encoding="utf-8") as f:
        payload = json.load(f)
    weights = payload.get("class_weights", payload)
    return {int(label): float(weight) for label, weight in weights.items()}


for required in [
    CACHE_DIR / "X_train.npy",
    CACHE_DIR / "y_train.npy",
    CACHE_DIR / "train_metadata.csv",
    CACHE_DIR / "X_validation.npy",
    CACHE_DIR / "y_validation.npy",
    CACHE_DIR / "validation_metadata.csv",
    CACHE_DIR / "feature_config.json",
    SHARED_CLASS_WEIGHT_PATH,
]:
    require_file(required)

X_train_scaled = np.load(CACHE_DIR / "X_train.npy")
y_train = np.load(CACHE_DIR / "y_train.npy")
train_meta = pd.read_csv(CACHE_DIR / "train_metadata.csv")
X_validation_scaled = np.load(CACHE_DIR / "X_validation.npy")
y_validation = np.load(CACHE_DIR / "y_validation.npy")
validation_meta = pd.read_csv(CACHE_DIR / "validation_metadata.csv")
mfcc_mean = np.load(CACHE_DIR / "mfcc_mean.npy")
mfcc_std = np.load(CACHE_DIR / "mfcc_std.npy")
with open(CACHE_DIR / "feature_config.json", "r", encoding="utf-8") as f:
    feature_config = json.load(f)
CLASS_WEIGHTS = load_shared_class_weights(SHARED_CLASS_WEIGHT_PATH)
class_weights = CLASS_WEIGHTS


X_test_scaled = np.load(CACHE_DIR / "X_test.npy")
y_test = np.load(CACHE_DIR / "y_test.npy")
test_meta = pd.read_csv(CACHE_DIR / "test_metadata.csv")
print("X_test shape:", X_test_scaled.shape)


print("Loaded LSTM-ready cache:", CACHE_DIR)
print("X_train shape:", X_train_scaled.shape)
print("X_validation shape:", X_validation_scaled.shape)
print("Shared class weights:", CLASS_WEIGHTS)


## 2. Final Selected Model and Test Evaluation


In [ ]:
# Purpose: The only optional LSTM notebook allowed to evaluate the held-out test set.
# The run flag is False by default so test metrics are produced only intentionally.
RUN_FINAL_TEST_EVALUATION = False
OPTIMISATION_DIR = OUTPUT_DIR / "optimisation"
selected_path = OPTIMISATION_DIR / "selected_hyperparameters.json"
test_metrics_path = METRICS_DIR / "lstm_test_metrics.json"
confusion_matrix_path = METRICS_DIR / "lstm_confusion_matrix.csv"
confusion_plot_path = FIGURES_DIR / "lstm_confusion_matrix.png"
final_model_path = MODELS_DIR / "lstm_final_selected_model.keras"


def build_model(config):
    model = Sequential()
    model.add(Input(shape=X_train_scaled.shape[1:]))
    model.add(LSTM(int(config["units"])))
    model.add(Dropout(float(config["dropout"])))
    model.add(Dense(1, activation="sigmoid"))
    optimizer = tf.keras.optimizers.Adam(learning_rate=float(config["learning_rate"]))
    model.compile(loss="binary_crossentropy", optimizer=optimizer, metrics=["accuracy"])
    return model


if RUN_FINAL_TEST_EVALUATION:
    require_file(selected_path)
    with open(selected_path, "r", encoding="utf-8") as f:
        selected = json.load(f)
    config = selected["config"]
    batch_size = int(config["batch_size"])
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(RANDOM_STATE)
    best_model = build_model(config)
    callbacks = [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)]
    history = best_model.fit(
        X_train_scaled,
        y_train,
        validation_data=(X_validation_scaled, y_validation),
        epochs=EPOCHS,
        batch_size=batch_size,
        class_weight=CLASS_WEIGHTS,
        callbacks=callbacks,
        verbose=2,
    )
    test_probability = best_model.predict(X_test_scaled, batch_size=batch_size, verbose=0).ravel()
    test_pred = (test_probability >= THRESHOLD).astype(int)
    test_metrics = {
        "model": "Optional LSTM",
        "input_representation": "T x 40 MFCC sequence",
        "selected_configuration": config,
        "test_accuracy": accuracy_score(y_test, test_pred),
        "test_precision": precision_score(y_test, test_pred, zero_division=0),
        "test_recall": recall_score(y_test, test_pred, zero_division=0),
        "test_f1": f1_score(y_test, test_pred, zero_division=0),
        "threshold": THRESHOLD,
    }
    with open(test_metrics_path, "w", encoding="utf-8") as f:
        json.dump(test_metrics, f, indent=2)
    cm = confusion_matrix(y_test, test_pred, labels=[0, 1])
    pd.DataFrame(cm, index=[CLASS_NAMES[0], CLASS_NAMES[1]], columns=[CLASS_NAMES[0], CLASS_NAMES[1]]).to_csv(confusion_matrix_path)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=[CLASS_NAMES[0], CLASS_NAMES[1]], yticklabels=[CLASS_NAMES[0], CLASS_NAMES[1]])
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.title("Optional LSTM final test confusion matrix")
    plt.tight_layout()
    plt.savefig(confusion_plot_path, dpi=300, bbox_inches="tight")
    plt.show()
    best_model.save(str(final_model_path))
    display(pd.DataFrame([test_metrics]))
else:
    print("RUN_FINAL_TEST_EVALUATION is False. Enable only after validation-only model selection is complete.")


## 3. Reproducibility Checks


In [ ]:
checks = {
    "final_test_evaluation_allowed_here": True,
    "loads_lstm_ready_cache": str(CACHE_DIR),
    "does_not_rescan_audio": True,
    "does_not_create_split": True,
    "does_not_extract_mfcc": True,
    "do_not_retune_after_test": True,
}
display(pd.DataFrame(list(checks.items()), columns=["check", "value"]))
